# Benchmark Results Visualization
Task 2.26-2.30: Visualize 5-variant benchmark results

## Objectives
- Generate benchmark comparison table
- Plot F1/Recall/FPR comparisons
- Throughput vs Memory scatter plot
- Statistical significance heatmap
- Export results for documentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Load Benchmark Results

In [ ]:
# Load results
results_path = '../experiments/benchmark_results/benchmark_results_latest.csv'

# Try to find latest results file
results_dir = Path('../experiments/benchmark_results')
if results_dir.exists():
    result_files = sorted(results_dir.glob('benchmark_results_*.csv'))
    if result_files:
        results_path = result_files[-1]
        print(f"Loading: {results_path}")

df = pd.read_csv(results_path)
print(f"Loaded {len(df)} runs")
print(f"\nVariants: {df['variant'].unique()}")
print(f"Seeds: {df['seed'].unique()}")

df.head()

## 2. Benchmark Comparison Table

In [ ]:
# Calculate summary statistics
summary = df.groupby('variant').agg({
    'f1': ['mean', 'std'],
    'recall': ['mean', 'std'],
    'fpr': ['mean', 'std'],
    'precision': ['mean', 'std'],
    'throughput_eps': ['mean', 'std'],
    'memory_mb': ['mean', 'std']
}).round(3)

# Format as table
def format_mean_std(row, metric):
    mean = row[(metric, 'mean')]
    std = row[(metric, 'std')]
    return f"{mean:.3f}±{std:.3f}"

table_df = pd.DataFrame({
    'Variant': summary.index,
    'F1': [format_mean_std(row, 'f1') for _, row in summary.iterrows()],
    'Recall': [format_mean_std(row, 'recall') for _, row in summary.iterrows()],
    'FPR': [format_mean_std(row, 'fpr') for _, row in summary.iterrows()],
    'Precision': [format_mean_std(row, 'precision') for _, row in summary.iterrows()],
    'Throughput (eps)': [format_mean_std(row, 'throughput_eps') for _, row in summary.iterrows()],
    'Memory (MB)': [format_mean_std(row, 'memory_mb') for _, row in summary.iterrows()]
})

print("\n" + "="*80)
print("BENCHMARK COMPARISON TABLE")
print("="*80)
print(table_df.to_string(index=False))
print("="*80)

# Save to CSV
table_df.to_csv('../experiments/benchmark_comparison_table.csv', index=False)
print("\n✅ Table saved: experiments/benchmark_comparison_table.csv")

## 3. Performance Metrics Comparison

In [ ]:
# Bar plot for F1, Recall, FPR
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['f1', 'recall', 'fpr']
titles = ['F1 Score', 'Recall (TPR)', 'False Positive Rate']

for ax, metric, title in zip(axes, metrics, titles):
    # Calculate mean and std for each variant
    summary_metric = df.groupby('variant')[metric].agg(['mean', 'std'])
    
    # Plot
    summary_metric['mean'].plot(kind='bar', ax=ax, yerr=summary_metric['std'], 
                                 capsize=5, color='steelblue', alpha=0.8)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Variant')
    ax.set_ylabel(title)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Highlight best
    if metric == 'fpr':
        best_idx = summary_metric['mean'].idxmin()
    else:
        best_idx = summary_metric['mean'].idxmax()
    
    bars = ax.patches
    best_bar_idx = list(summary_metric.index).index(best_idx)
    bars[best_bar_idx].set_color('green')
    bars[best_bar_idx].set_alpha(1.0)

plt.tight_layout()
plt.savefig('../docs/figures/benchmark_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: docs/figures/benchmark_metrics_comparison.png")

## 4. Throughput vs Memory Scatter Plot

In [ ]:
# Scatter plot: Throughput vs Memory
fig, ax = plt.subplots(figsize=(10, 8))

# Calculate means per variant
variant_means = df.groupby('variant').agg({
    'throughput_eps': 'mean',
    'memory_mb': 'mean',
    'f1': 'mean'
})

# Scatter plot
scatter = ax.scatter(
    variant_means['throughput_eps'],
    variant_means['memory_mb'],
    s=variant_means['f1'] * 1000,  # Size by F1 score
    alpha=0.6,
    c=range(len(variant_means)),
    cmap='viridis'
)

# Annotate points
for variant, row in variant_means.iterrows():
    ax.annotate(
        variant,
        (row['throughput_eps'], row['memory_mb']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
    )

ax.set_xlabel('Throughput (events/sec)', fontsize=12)
ax.set_ylabel('Memory Usage (MB)', fontsize=12)
ax.set_title('Throughput vs Memory (bubble size = F1 score)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

# Ideal region (high throughput, low memory)
ax.axhline(y=variant_means['memory_mb'].median(), color='red', linestyle='--', alpha=0.3, label='Median Memory')
ax.axvline(x=variant_means['throughput_eps'].median(), color='blue', linestyle='--', alpha=0.3, label='Median Throughput')
ax.legend()

plt.tight_layout()
plt.savefig('../docs/figures/throughput_vs_memory.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: docs/figures/throughput_vs_memory.png")

## 5. Statistical Significance Results

In [ ]:
# Load statistical test results if available
stat_test_path = Path('../experiments/statistical_test_results.csv')

if stat_test_path.exists():
    stat_df = pd.read_csv(stat_test_path)
    
    # Pivot for heatmap
    pivot = stat_df.pivot_table(
        index='comparison',
        columns='metric',
        values='p_value'
    )
    
    # Create significance mask (p < 0.05)
    significance_mask = pivot < 0.05
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.4f',
        cmap='RdYlGn_r',
        vmin=0,
        vmax=0.1,
        cbar_kws={'label': 'p-value'},
        ax=ax,
        linewidths=0.5
    )
    
    # Add significance markers
    for i in range(len(pivot)):
        for j in range(len(pivot.columns)):
            if significance_mask.iloc[i, j]:
                ax.text(
                    j + 0.5, i + 0.7,
                    '✓',
                    ha='center',
                    va='center',
                    color='darkgreen',
                    fontsize=16,
                    fontweight='bold'
                )
    
    ax.set_title('Statistical Significance (p-values)\n✓ = Significant (p < 0.05)', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Metric')
    ax.set_ylabel('Variant Comparison (vs proposed_context_aware)')
    
    plt.tight_layout()
    plt.savefig('../docs/figures/statistical_significance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: docs/figures/statistical_significance.png")
else:
    print("⚠ Statistical test results not found. Run experiments/statistical_tests.py first.")

## 6. Export Summary for Documentation

In [ ]:
# Create markdown summary
markdown_summary = f"""
# Benchmark Results Summary

**Date:** {pd.Timestamp.now().strftime('%Y-%m-%d')}

## Overview
- **Total Runs:** {len(df)}
- **Variants:** {len(df['variant'].unique())}
- **Seeds:** {len(df['seed'].unique())}

## Comparison Table

{table_df.to_markdown(index=False)}

## Key Findings

### Best Performance
- **Best F1:** {table_df.loc[table_df['F1'].apply(lambda x: float(x.split('±')[0])).idxmax(), 'Variant']}
- **Best Recall:** {table_df.loc[table_df['Recall'].apply(lambda x: float(x.split('±')[0])).idxmax(), 'Variant']}
- **Lowest FPR:** {table_df.loc[table_df['FPR'].apply(lambda x: float(x.split('±')[0])).idxmin(), 'Variant']}
- **Highest Throughput:** {table_df.loc[table_df['Throughput (eps)'].apply(lambda x: float(x.split('±')[0])).idxmax(), 'Variant']}

## Figures
- Performance Metrics: `docs/figures/benchmark_metrics_comparison.png`
- Throughput vs Memory: `docs/figures/throughput_vs_memory.png`
- Statistical Significance: `docs/figures/statistical_significance.png`
"""

# Save markdown
with open('../experiments/benchmark_summary.md', 'w') as f:
    f.write(markdown_summary)

print(markdown_summary)
print("\n✅ Summary saved: experiments/benchmark_summary.md")

## Conclusion

This notebook provides comprehensive visualization and analysis of the 5-variant benchmark results.

**Next Steps:**
1. Review statistical significance results
2. Document final model selection
3. Prepare results for Phase 3